# MATCARE-AI: General Maternal Health Risk Analysis & Hybrid Modeling
This notebook executes the exploratory data analysis (EDA), anomaly cleaning,
stratified train/val/test splitting, base model training (Random Forest & XGBoost),
validation-based adaptive probability weight optimization, and SHAP explainability.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append('..')
from preprocessing.general_risk_preprocessor import load_and_inspect_general_risk, get_general_risk_split
from training.hybrid_model import AdaptiveWeightedHybridClassifier
from evaluation.metrics import compare_four_approaches
from explainability.shap_explainers import compute_tree_shap_importances, compute_hybrid_global_importances

## 1. Load Dataset & Perform Exploratory Data Analysis

In [ ]:
data_path = os.path.join('..', 'datasets', 'maternal_health_risk.csv')
df = load_and_inspect_general_risk(data_path)
print('Dataset Shape:', df.shape)
display(df.head())
display(df.describe().round(2))
print('Risk Level Counts:\n', df['RiskLevelClean'].value_counts())

## 2. Physiological Outlier Inspection & Stratified Splitting

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, scaler, metadata = get_general_risk_split(df)
print(f'Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}')
print('Class distribution preserved across splits.')

## 3. Train Base Models & Optimize Adaptive Voting Weights on Validation Set

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train)

hybrid = AdaptiveWeightedHybridClassifier(rf_model=rf, xgb_model=xgb, classes=np.array([0, 1, 2]))
opt_res = hybrid.optimize_weights(X_val, y_val, metric='clinical_utility', high_risk_idx=2)
print(f'Optimal Learned Weights: w_RF={opt_res["best_w_rf"]:.4f}, w_XGB={opt_res["best_w_xgb"]:.4f}')

## 4. Evaluate Held-Out Test Set: Comparing 4 Approaches

In [ ]:
comparison = compare_four_approaches(rf, xgb, hybrid, X_test, y_test, metadata['class_names'], high_risk_idx=2)
comp_df = pd.DataFrame([{
    'Model': c['model_name'],
    'Accuracy (%)': c['accuracy'],
    'Macro F1 (%)': c['macro_f1'],
    'High-Risk Recall (%)': c.get('high_risk_recall', 0),
    'High-Risk FPR (%)': c.get('high_risk_fpr', 0),
    'ROC-AUC': c['roc_auc']
} for c in comparison])
display(comp_df)

## 5. SHAP Explainability & Global Feature Importance

In [ ]:
rf_imp = compute_tree_shap_importances(rf, X_val, metadata['feature_names'])
xgb_imp = compute_tree_shap_importances(xgb, X_val, metadata['feature_names'])
hybrid_imp = compute_hybrid_global_importances(rf_imp, xgb_imp, opt_res['best_w_rf'], opt_res['best_w_xgb'], metadata['feature_names'])

plt.figure(figsize=(10, 5))
sns.barplot(x=list(hybrid_imp.values()), y=list(hybrid_imp.keys()), palette='viridis')
plt.title('Hybrid SHAP Global Feature Importance (UCI Maternal Risk)')
plt.xlabel('Importance Proportion')
plt.show()